# Window reset and warmup tests

Executable definitions and evidence checks. Historical evidence is identified separately from new runs.


In [1]:
from pathlib import Path
import importlib.abc, importlib.util, json, sys
TRACE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/notebook_execution.yaml').exists())
MODULES = {'contract':'runtime_contract','baseline':'environment_adapter','runtime':'unity_runtime'}
class NotebookModules(importlib.abc.MetaPathFinder, importlib.abc.Loader):
    def find_spec(self, fullname, path=None, target=None):
        name = MODULES.get(fullname, fullname)
        candidate = TRACE_ROOT/'notebooks/library'/f'{name}.ipynb'
        if '.' not in fullname and candidate.is_file():
            return importlib.util.spec_from_loader(fullname, self, origin=str(candidate))
    def create_module(self, spec): return None
    def exec_module(self, module):
        path=Path(module.__spec__.origin)
        module.__file__=str(path); module.TRACE_ROOT=TRACE_ROOT
        for i,cell in enumerate(json.loads(path.read_text())['cells']):
            if cell['cell_type']=='code' and 'module' in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+f':cell-{i+1}','exec'),module.__dict__)
if not any(type(x).__name__=='NotebookModules' for x in sys.meta_path):
    sys.meta_path.insert(0,NotebookModules())
print('Notebook module loader ready:', TRACE_ROOT.name)


Notebook module loader ready: trace-lab


In [2]:
import unittest,tempfile,json
from pathlib import Path
import numpy as np
from environment_adapter import BaselineState
from baseline_rewards import TaskReward,MaxSupportReward
class Checks(unittest.TestCase):
    def test_real_windows_reset_and_horizon(self):
        pairs=[]
        def model(pair):pairs.append(pair.copy());return .6,{}
        state=BaselineState(model,'MAX');state.reset(np.zeros(81))
        for i in range(1,601):
            obs,reward,term,trunc,info=state.step(np.full(81,i),0)
            self.assertEqual(info['fresh'],i>=30 and i%15==0)
            self.assertEqual(reward,.6 if info['fresh'] else 0)
        self.assertEqual(len(pairs),39);self.assertTrue(trunc);self.assertFalse(term)
        np.testing.assert_allclose(pairs[0][:27],8);np.testing.assert_allclose(pairs[0][27:],23)
        with self.assertRaises(RuntimeError):state.step(np.zeros(81),0)
        obs=state.reset(np.zeros(81));np.testing.assert_array_equal(obs[-5:],0)
        for i in range(29):self.assertFalse(state.step(np.zeros(81),0)[4]['valid'])
        self.assertEqual(len(pairs),39)
print('Window reset and warmup tests definitions/execution completed.')


Frozen increasing-preference scoring definitions/execution completed.
Task progress and fresh-pair rewards definitions/execution completed.
Matched observation action and window adapter definitions/execution completed.
Window reset and warmup tests definitions/execution completed.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
suite=unittest.defaultTestLoader.loadTestsFromTestCase(Checks)
result=unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()
print('Tests run:',result.testsRun)

test_real_windows_reset_and_horizon (__main__.Checks) ... 

ok


----------------------------------------------------------------------
Ran 1 test in 0.014s

OK


Tests run: 1
